# County-Level Nighttime Light Preprocessing

This notebook preprocesses **VIIRS Nighttime Lights (VNL) data** by clipping the global raster to individual US counties and saving each county as a separate GeoTIFF and PNG image.

**Data sources:**
- [VIIRS VNL 2024 Annual Global](https://eogdata.mines.edu/nighttime_light/) — `VNL_npp_2024_global_vcmslcfg_v2_c202502261200.average_masked.dat.tif`
- [TIGER/Line US County Shapefile 2024](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html) — `tl_2024_us_county`

**Pipeline Overview:**
1. Load & preview global nighttime light raster
2. Clip to continental USA bounding box
3. Load US county boundaries
4. Clip raster per county → save as `.tif`
5. Render county images as grayscale PNG (transparent background)
6. Render county images as grayscale PNG (black background)


## 0. Setup & Configuration

Set `ROOT_DIR` to the root of your project folder and `DATA_DIR` to where the raw data files are stored.

In [ ]:
import os

# ─── Configure paths here ───────────────────────────────────────────────────
ROOT_DIR = os.path.abspath(".")                  # project root (change if needed)
DATA_DIR = os.path.join(ROOT_DIR, "data")        # raw data folder

VNL_FILE  = os.path.join(DATA_DIR, "VNL_npp_2024_global_vcmslcfg_v2_c202502261200.average_masked.dat.tif")
COUNTY_SHP = os.path.join(DATA_DIR, "tl_2024_us_county")

OUTPUT_DIR      = os.path.join(ROOT_DIR, "county_outputs")          # per-county GeoTIFFs + labels.csv
OVERLAY_DIR     = os.path.join(ROOT_DIR, "county_overlay_gray_png") # transparent-bg PNGs
DARK_PNG_DIR    = os.path.join(ROOT_DIR, "county_dark_png")         # black-bg PNGs
# ────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR,   exist_ok=True)
os.makedirs(OVERLAY_DIR,  exist_ok=True)
os.makedirs(DARK_PNG_DIR, exist_ok=True)

print("ROOT_DIR :", ROOT_DIR)
print("VNL_FILE :", VNL_FILE)
print("COUNTY_SHP:", COUNTY_SHP)


## 1. Load & Preview Global Nighttime Light Raster

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from rasterio.windows import Window

src = rasterio.open(VNL_FILE)
print(src)

# Quick sanity-check: visualize a small window
window = Window(15000, 8000, 1000, 1000)
img = src.read(1, window=window)

log_img = np.log1p(img)
vmax = np.percentile(log_img, 99)

plt.imshow(log_img, cmap='inferno', vmax=vmax)
plt.colorbar()
plt.title("Nighttime Light (Scaled)")
plt.show()


## 2. Clip to Continental USA Bounding Box

In [ ]:
from shapely.geometry import box
import geopandas as gpd
from rasterio.mask import mask

# Continental USA bounding box
min_lon, min_lat = -125, 24
max_lon, max_lat =  -66, 50

bbox = box(min_lon, min_lat, max_lon, max_lat)
gdf  = gpd.GeoDataFrame({'geometry': [bbox]}, crs="EPSG:4326").to_crs(src.crs)

out_img, _ = mask(src, [gdf.iloc[0].geometry], crop=True)

log_img = np.log1p(out_img[0])
vmax    = np.percentile(log_img, 99)

plt.figure(figsize=(10, 6))
plt.imshow(log_img, cmap='inferno', vmax=vmax)
plt.title("USA Nighttime Lights")
plt.axis('off')

save_path = os.path.join(ROOT_DIR, "usa_nighttime_lights.png")
plt.savefig(save_path, bbox_inches='tight', pad_inches=0, dpi=300)
plt.show()
print(f"Saved: {save_path}")


## 3. Load US County Boundaries

In [ ]:
# FIPS code → state name mapping (all 50 states + DC)
FIPS_TO_STATE = {
    "01": "Alabama",         "02": "Alaska",        "04": "Arizona",
    "05": "Arkansas",        "06": "California",     "08": "Colorado",
    "09": "Connecticut",     "10": "Delaware",       "11": "District of Columbia",
    "12": "Florida",         "13": "Georgia",        "15": "Hawaii",
    "16": "Idaho",           "17": "Illinois",       "18": "Indiana",
    "19": "Iowa",            "20": "Kansas",         "21": "Kentucky",
    "22": "Louisiana",       "23": "Maine",          "24": "Maryland",
    "25": "Massachusetts",   "26": "Michigan",       "27": "Minnesota",
    "28": "Mississippi",     "29": "Missouri",       "30": "Montana",
    "31": "Nebraska",        "32": "Nevada",         "33": "New Hampshire",
    "34": "New Jersey",      "35": "New Mexico",     "36": "New York",
    "37": "North Carolina",  "38": "North Dakota",   "39": "Ohio",
    "40": "Oklahoma",        "41": "Oregon",         "42": "Pennsylvania",
    "44": "Rhode Island",    "45": "South Carolina", "46": "South Dakota",
    "47": "Tennessee",       "48": "Texas",          "49": "Utah",
    "50": "Vermont",         "51": "Virginia",       "53": "Washington",
    "54": "West Virginia",   "55": "Wisconsin",      "56": "Wyoming",
}

county = gpd.read_file(COUNTY_SHP)
county["state_fips"] = county["GEOID"].str[:2]
county["state"]      = county["state_fips"].map(FIPS_TO_STATE).fillna("Unknown")

print(f"Loaded {len(county)} counties")

# Quick visual check
fig, ax = plt.subplots(figsize=(6, 6))
county[county['NAME'] == "Hennepin"].plot(ax=ax)
plt.title("Hennepin County")
plt.savefig(os.path.join(ROOT_DIR, "hennepin_county.png"), bbox_inches='tight', pad_inches=0, dpi=300)
plt.show()


## 4. Clip Raster per County → Save as GeoTIFF + labels.csv

In [ ]:
import pandas as pd

county_proj = county.to_crs(src.crs)
county_proj["geometry"] = county_proj["geometry"].buffer(0)  # fix self-intersections

records = []

for i, row in county_proj.iterrows():
    geom   = [row.geometry]
    name   = str(row["NAME"]).replace(" ", "_").replace("/", "_")
    geoid  = str(row.get("GEOID", i))
    fips   = str(row.get("state_fips", i))

    try:
        clipped, transform = mask(src, geom, crop=True)

        out_meta = src.meta.copy()
        out_meta.update({
            "driver":    "GTiff",
            "height":    clipped.shape[1],
            "width":     clipped.shape[2],
            "transform": transform,
        })

        filename = f"{fips}_{geoid}_{name}.tif"
        out_path = os.path.join(OUTPUT_DIR, filename)

        with rasterio.open(out_path, "w", **out_meta) as dest:
            dest.write(clipped)
            dest.update_tags(NAME=row["NAME"], GEOID=geoid)

        records.append({
            "filename":   filename,
            "NAME":       row["NAME"],
            "GEOID":      geoid,
            "STATE":      row["state"],
            "STATE_FIPS": row["state_fips"],
        })

    except Exception as e:
        print(f"Error at {i} ({row['NAME']}): {e}")

label_df = pd.DataFrame(records)
label_df.to_csv(os.path.join(OUTPUT_DIR, "labels.csv"), index=False)
print(f"Done. {len(label_df)} counties saved to {OUTPUT_DIR}")
print(label_df.head())


## 5. Render County PNGs — Transparent Background (Grayscale)

Each county TIF is rendered as a grayscale image with a black county mask and a transparent background.

In [ ]:
from rasterio.plot import plotting_extent

county_proj = county.to_crs(src.crs)

for file in os.listdir(OUTPUT_DIR):
    if not file.endswith(".tif"):
        continue

    parts      = file.replace(".tif", "").split("_")
    state_fip  = parts[0]
    geoid      = parts[1]
    county_name = "_".join(parts[2:])

    tif_path = os.path.join(OUTPUT_DIR, file)
    png_path = os.path.join(OVERLAY_DIR, file.replace(".tif", ".png"))

    county_poly = county_proj[county_proj["GEOID"].astype(str) == str(geoid)]

    with rasterio.open(tif_path) as ds:
        img    = ds.read(1).astype(float)
        extent = plotting_extent(ds)

    img_masked = np.ma.masked_where(img <= 0, img)
    log_img    = np.log1p(img_masked)

    height, width = img.shape
    dpi   = 100
    fig_w = width  / dpi
    fig_h = height / dpi

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.patch.set_alpha(0)
    ax.patch.set_alpha(0)
    ax.set_position([0, 0, 1, 1])

    county_poly.plot(ax=ax, facecolor="black", edgecolor="none", alpha=1.0, zorder=1)

    cmap = plt.cm.gray.copy()
    cmap.set_bad(alpha=0)

    ax.imshow(log_img, cmap=cmap, vmin=0, vmax=5, extent=extent, zorder=2, aspect='auto')

    left, right, bottom, top = extent
    ax.set_xlim(left, right)
    ax.set_ylim(bottom, top)
    ax.set_aspect("equal", adjustable="box")
    ax.axis("off")

    plt.savefig(png_path, dpi=dpi, transparent=True, pad_inches=0)
    plt.close()

print(f"Saved transparent PNGs to: {OVERLAY_DIR}")


## 6. Render County PNGs — Black Background (Grayscale)

Same as above but with a solid black background instead of transparency.

In [ ]:
for file in os.listdir(OUTPUT_DIR):
    if not file.endswith(".tif"):
        continue

    state_fip   = file.split("_")[0]
    tif_path    = os.path.join(OUTPUT_DIR, file)
    png_path    = os.path.join(DARK_PNG_DIR, file.replace(".tif", ".png"))

    with rasterio.open(tif_path) as ds:
        img           = ds.read(1).astype(float)
        tif_w, tif_h  = ds.width, ds.height

    img_masked = np.ma.masked_where(img <= 0, img)
    log_img    = np.log1p(img_masked)

    cmap = plt.cm.gray.copy()
    cmap.set_bad(color='black')

    dpi   = 100
    fig_w = tif_w / dpi
    fig_h = tif_h / dpi

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    fig.patch.set_facecolor('black')
    ax.set_facecolor('black')

    ax.imshow(log_img, cmap=cmap, vmin=0, vmax=5)
    ax.axis("off")

    plt.savefig(png_path, bbox_inches='tight', pad_inches=0, dpi=dpi, facecolor='black')
    plt.close()

print(f"Saved dark PNGs to: {DARK_PNG_DIR}")
